# Graph Representations for Molecule Analysis

This notebook turns the abstract graph-theory requirements into concrete graph representations built from the available molecular datasets.

Representations in this notebook:
- a toy atom-bond graph
- a molecule-assay bipartite graph from Tox21
- a molecule-similarity graph from BBBP SMILES strings

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid")

In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == 'notebooks' else cwd


PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
bbbp = pd.read_csv(DATA_DIR / 'BBBP.csv')
tox21 = pd.read_csv(DATA_DIR / 'tox21.csv')
print('BBBP shape:', bbbp.shape)
print('Tox21 shape:', tox21.shape)

## 1. Atom-Bond Graph as Matrices

A molecular graph begins with atoms as nodes and bonds as edges.

In [ ]:
atom_labels = ['C1', 'C2', 'N3', 'O4', 'H5']
adjacency = np.array([
    [0, 1, 1, 0, 1],
    [1, 0, 0, 1, 0],
    [1, 0, 0, 1, 0],
    [0, 1, 1, 0, 0],
    [1, 0, 0, 0, 0],
], dtype=float)
degree = adjacency.sum(axis=1)
laplacian = np.diag(degree) - adjacency

display(pd.DataFrame(adjacency, index=atom_labels, columns=atom_labels))
print('Degree sequence:', degree.astype(int).tolist())
print('Laplacian eigenvalues:', np.round(np.linalg.eigvalsh(laplacian), 4).tolist())

## 2. Molecule-Assay Bipartite Graph

For Tox21, molecules connect to assays when a label is active. This creates a bipartite graph with molecule nodes on one side and assay nodes on the other.

In [ ]:
sample = tox21.head(30).copy()
assay_columns = [column for column in tox21.columns if column not in {'mol_id', 'smiles'}]
bipartite_matrix = sample[assay_columns].fillna(0).astype(int)
activity_per_molecule = bipartite_matrix.sum(axis=1)
activity_per_assay = bipartite_matrix.sum(axis=0).sort_values(ascending=False)

display(activity_per_assay.head(12).rename_axis('assay').reset_index(name='active_edges'))

plt.figure(figsize=(10, 5))
sns.barplot(x=activity_per_assay.head(12).values, y=activity_per_assay.head(12).index)
plt.title('Active Tox21 Assay Edges in a 30-Molecule Sample')
plt.xlabel('Active edges')
plt.ylabel('Assay')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(bipartite_matrix.iloc[:15, :12], cmap='crest', cbar=False)
plt.title('Tox21 Bipartite Incidence Matrix Sample')
plt.xlabel('Assays')
plt.ylabel('Molecules')
plt.show()

## 3. Molecule-Similarity Graph from BBBP SMILES

Without RDKit, we can still build a useful graph by connecting molecules whose character n-gram profiles are similar.

In [ ]:
similarity_sample = bbbp[['name', 'smiles', 'p_np']].head(20).copy()
vectorizer = CountVectorizer(analyzer='char', ngram_range=(2, 4), min_df=1)
feature_matrix = vectorizer.fit_transform(similarity_sample['smiles'])
similarity = cosine_similarity(feature_matrix)
similarity_graph = (similarity >= 0.45).astype(int)
np.fill_diagonal(similarity_graph, 0)
degree_df = pd.DataFrame({
    'molecule': similarity_sample['name'].fillna(similarity_sample.index.astype(str)),
    'degree': similarity_graph.sum(axis=1),
    'target': similarity_sample['p_np'],
})
display(degree_df.head(10))

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(similarity_graph, cmap='mako', cbar=False)
plt.title('BBBP Similarity Graph Adjacency Matrix')
plt.xlabel('Molecules')
plt.ylabel('Molecules')
plt.show()

plt.figure(figsize=(10, 4))
sns.barplot(data=degree_df.sort_values('degree', ascending=False).head(10), x='degree', y='molecule', hue='target')
plt.title('Highest-Degree Molecules in the BBBP Similarity Graph')
plt.xlabel('Graph degree')
plt.ylabel('Molecule')
plt.show()

## Why These Graph Views Matter

These three representations map directly onto later modeling work:
- atom-bond graphs lead to message passing and GNNs
- molecule-assay bipartite graphs lead to link prediction and multitask structure
- similarity graphs lead to neighborhood methods and graph regularization